## Full Adipocytes AnnData 
### subset to genes in GeneFormer (to not run into tokenisation problems and shape mismatches) 

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
import matplotlib.pyplot as plt
import os
import anndata 
import gc
from scipy.sparse import csr_matrix

seed=42
np.random.seed(seed)


main_path = "/rds/general/user/ap5625/home/"

input_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/T2D_REM/Phase_2/objects"

output_path = "/rds/general/user/ap5625/projects/lms-scott-raw/live/Ada/perturbation_modeling"

In [2]:
adata = sc.read_h5ad(input_path + "/Adipocytes_annotated_step2.h5ad")

In [4]:
import pickle

# Load the Geneformer gene→token mapping - they have it in their code 
with open('/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl', 'rb') as f:
    gf_genes = pickle.load(f)


In [5]:
# Keep only genes present in the Geneformer vocabulary
before = adata.n_vars


In [6]:
adata.var['in_gf'] = adata.var_names.isin(gf_genes.values()) | adata.var_names.isin(gf_genes.keys())
adata = adata[:, adata.var['in_gf']].copy()


In [7]:
print(f"Genes: {before} → {adata.n_vars} after Geneformer filtering")

Genes: 38606 → 38291 after Geneformer filtering


# For a gene to end up in the tokenisation dictionary 
## Expression based filtering 
## Non-zero median requirement 

In [8]:
# also genes with very low expression (near zero) - do not get added in the tokenisation dictionary 

# On your Geneformer-filtered adata (raw counts in X or layers['raw'])
counts = adata.layers['raw'] if 'raw' in adata.layers else adata.X
genes_expressed = np.asarray((counts > 0).sum(axis=0)).flatten() > 0
print(f"Genes with non-zero expression in ≥1 cell: {genes_expressed.sum()} / {adata.n_vars}")

# Genes expressed in a meaningful number of cells
genes_10cells = np.asarray((counts > 0).sum(axis=0)).flatten() >= 10
print(f"Genes expressed in ≥10 cells: {genes_10cells.sum()} / {adata.n_vars}")

Genes with non-zero expression in ≥1 cell: 35724 / 38291
Genes expressed in ≥10 cells: 32046 / 38291
